# Instance-Aware Object Removal: Single-Shot vs Progressive Inpainting
## CSE-498R Directed Research — Week 4 Experiment

**Workflow:**
1. Upload an image → YOLO detects persons → SAM generates masks
2. **Method A (Single-Shot):** Merge all masks → inpaint once
3. **Method B (Progressive):** Remove persons one-by-one (largest-first), each inpainted result feeds the next step
4. Compare with PSNR, SSIM, LPIPS + visual grids
5. Repeat for as many images as you want — summary table accumulates across all uploads

## Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install ultralytics segment-anything diffusers transformers accelerate lpips scikit-image opencv-python-headless Pillow torch torchvision -q

import os
if not os.path.exists('sam_vit_h_4b8939.pth'):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    print('SAM checkpoint downloaded.')
else:
    print('SAM checkpoint already exists.')

## Cell 2 — Imports & Model Loading

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
from pathlib import Path
import warnings, io, os, time
warnings.filterwarnings('ignore')

from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
import lpips

from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor
from diffusers import StableDiffusionInpaintPipeline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

print('Loading YOLOv8x...')
yolo_model = YOLO('yolov8x.pt')

print('Loading SAM ViT-H...')
sam = sam_model_registry['vit_h'](checkpoint='sam_vit_h_4b8939.pth')
sam.to(DEVICE)
sam_predictor = SamPredictor(sam)

print('Loading Stable Diffusion Inpainting...')
inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    'runwayml/stable-diffusion-inpainting',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    safety_checker=None,
).to(DEVICE)
inpaint_pipe.set_progress_bar_config(disable=True)

print('Loading LPIPS...')
lpips_fn = lpips.LPIPS(net='alex').to(DEVICE)

print('\n All models loaded!')

## Cell 3 — Core Pipeline Functions

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
INPAINT_SIZE     = 512
INPAINT_PROMPT   = "empty background, realistic background, no people, photorealistic, seamless"
INPAINT_STEPS    = 30
INPAINT_GUIDANCE = 7.5


# ── Detection ─────────────────────────────────────────────────────────────────
def detect_persons(image_bgr, conf_thresh=0.4):
    """YOLO person detection → list of dicts sorted largest-area-first."""
    results = yolo_model(image_bgr, conf=conf_thresh, classes=[0], verbose=False)[0]
    boxes = []
    if results.boxes is not None:
        for box in results.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            boxes.append({'bbox': [x1, y1, x2, y2], 'area': (x2-x1)*(y2-y1)})
    boxes.sort(key=lambda b: b['area'], reverse=True)
    return boxes


# ── SAM Segmentation ──────────────────────────────────────────────────────────
def get_sam_mask(image_rgb, bbox):
    """SAM mask for a single bbox. Returns binary uint8 (H,W) 0/255."""
    sam_predictor.set_image(image_rgb)
    masks, scores, _ = sam_predictor.predict(
        point_coords=None, point_labels=None,
        box=np.array(bbox)[None, :], multimask_output=True
    )
    mask = masks[np.argmax(scores)].astype(np.uint8) * 255
    # Dilate slightly to cover edges
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    return cv2.dilate(mask, kernel, iterations=1)


def feather_mask(mask, blur_radius=21):
    """Gaussian-feathered soft-edge mask."""
    feathered = cv2.GaussianBlur(mask.astype(np.float32), (blur_radius, blur_radius), 0)
    return (feathered > 127).astype(np.uint8) * 255


# ── SD Inpainting ─────────────────────────────────────────────────────────────
def sd_inpaint(image_rgb, mask):
    """Inpaint masked region. Returns RGB numpy array same size as input."""
    H, W = image_rgb.shape[:2]
    pil_img  = Image.fromarray(image_rgb).resize((INPAINT_SIZE, INPAINT_SIZE), Image.LANCZOS)
    pil_mask = Image.fromarray(mask).resize((INPAINT_SIZE, INPAINT_SIZE), Image.NEAREST)

    with torch.autocast(DEVICE):
        out = inpaint_pipe(
            prompt=INPAINT_PROMPT, image=pil_img, mask_image=pil_mask,
            num_inference_steps=INPAINT_STEPS, guidance_scale=INPAINT_GUIDANCE,
        ).images[0]

    out_np = np.array(out.resize((W, H), Image.LANCZOS))
    mask_3 = np.stack([mask / 255.0] * 3, axis=-1)
    return (out_np * mask_3 + image_rgb * (1 - mask_3)).astype(np.uint8)


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(original, reconstructed, mask):
    m = mask > 127
    if not m.any():
        return {'PSNR': 0.0, 'SSIM': 0.0, 'LPIPS': 1.0, 'mask_coverage_%': 0.0}
    ys, xs = np.where(m)
    y1, y2, x1, x2 = ys.min(), ys.max(), xs.min(), xs.max()
    orig_c  = original[y1:y2, x1:x2]
    recon_c = reconstructed[y1:y2, x1:x2]

    psnr_val = psnr_fn(orig_c, recon_c, data_range=255)
    ssim_val = ssim_fn(orig_c, recon_c, channel_axis=2, data_range=255)

    def to_t(img):
        return (torch.from_numpy(img).float()/127.5-1).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        lp = lpips_fn(to_t(orig_c), to_t(recon_c)).item()

    return {
        'PSNR':  round(psnr_val, 3),
        'SSIM':  round(ssim_val, 4),
        'LPIPS': round(lp, 4),
        'mask_coverage_%': round(100.0 * m.sum() / m.size, 2),
    }

## Cell 4 — Two Removal Methods

In [ ]:
def method_single_shot(image_rgb, persons):
    """METHOD A: merge all masks → one inpainting call."""
    H, W = image_rgb.shape[:2]
    union_mask = np.zeros((H, W), dtype=np.uint8)
    print(f'  [A] Generating {len(persons)} SAM masks...')
    for p in persons:
        union_mask = np.maximum(union_mask, get_sam_mask(image_rgb, p['bbox']))
    union_mask = feather_mask(union_mask)
    print(f'  [A] Inpainting (mask covers {100*union_mask.astype(bool).sum()/union_mask.size:.1f}% of image)...')
    t0 = time.time()
    result = sd_inpaint(image_rgb, union_mask)
    return {'result': result, 'union_mask': union_mask, 'time_s': round(time.time()-t0, 1)}


def method_progressive(image_rgb, persons):
    """METHOD B: remove one-by-one largest-first; each result feeds the next step."""
    H, W = image_rgb.shape[:2]
    current  = image_rgb.copy()
    cum_mask = np.zeros((H, W), dtype=np.uint8)
    steps    = []
    t0 = time.time()
    for i, p in enumerate(persons):
        print(f'  [B] Step {i+1}/{len(persons)} — person area={p["area"]}px...')
        step_mask = feather_mask(get_sam_mask(current, p['bbox']))
        cum_mask  = np.maximum(cum_mask, step_mask)
        current   = sd_inpaint(current, step_mask)
        steps.append(current.copy())
    return {'result': current, 'union_mask': cum_mask,
            'step_images': steps, 'time_s': round(time.time()-t0, 1)}

## Cell 5 — Visualization

In [ ]:
def overlay_detections(image_rgb, persons):
    vis = image_rgb.copy()
    for i, p in enumerate(persons):
        x1,y1,x2,y2 = p['bbox']
        cv2.rectangle(vis, (x1,y1),(x2,y2),(0,230,80),3)
        cv2.putText(vis, f'P{i+1}', (x1+5,y1+35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0,230,80), 2)
    return vis


def show_results(image_name, original, persons, res_A, res_B, mA, mB, save_dir='results'):
    os.makedirs(save_dir, exist_ok=True)
    stem = Path(image_name).stem

    # ── 4-panel comparison ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    fig.suptitle(f'{image_name}  |  {len(persons)} person(s) detected', fontsize=13, fontweight='bold')

    def show(ax, img, title, sub=''):
        ax.imshow(img); ax.set_title(f'{title}\n{sub}', fontsize=9); ax.axis('off')

    show(axes[0], original, 'Original')
    show(axes[1], overlay_detections(original, persons), 'YOLO Detections', f'{len(persons)} person(s)')
    show(axes[2], res_A['result'], 'Method A — Single-Shot',
         f"PSNR={mA['PSNR']} | SSIM={mA['SSIM']} | LPIPS={mA['LPIPS']}\n"
         f"Mask={mA['mask_coverage_%']}% | {res_A['time_s']}s")
    show(axes[3], res_B['result'], 'Method B — Progressive',
         f"PSNR={mB['PSNR']} | SSIM={mB['SSIM']} | LPIPS={mB['LPIPS']}\n"
         f"Mask={mB['mask_coverage_%']}% | {res_B['time_s']}s")
    plt.tight_layout()
    p1 = f'{save_dir}/{stem}_comparison.png'
    plt.savefig(p1, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {p1}')

    # ── Progressive step strip ────────────────────────────────────────────
    steps = res_B.get('step_images', [])
    if len(steps) > 1:
        fig2, axes2 = plt.subplots(1, len(steps)+1, figsize=(6*(len(steps)+1), 5))
        fig2.suptitle('Method B — Progressive Steps (Largest-First)', fontsize=12, fontweight='bold')
        axes2[0].imshow(original); axes2[0].set_title('Original'); axes2[0].axis('off')
        for j, s in enumerate(steps):
            axes2[j+1].imshow(s); axes2[j+1].set_title(f'After P{j+1}'); axes2[j+1].axis('off')
        plt.tight_layout()
        p2 = f'{save_dir}/{stem}_progressive_steps.png'
        plt.savefig(p2, dpi=130, bbox_inches='tight'); plt.show()
        print(f'Saved: {p2}')

    # ── Mask overlay ──────────────────────────────────────────────────────
    fig3, axes3 = plt.subplots(1, 2, figsize=(12, 5))
    axes3[0].imshow(original); axes3[0].imshow(res_A['union_mask'], alpha=0.5, cmap='Reds')
    axes3[0].set_title('Method A — Union Mask'); axes3[0].axis('off')
    axes3[1].imshow(original); axes3[1].imshow(res_B['union_mask'], alpha=0.5, cmap='Blues')
    axes3[1].set_title('Method B — Cumulative Mask'); axes3[1].axis('off')
    plt.tight_layout()
    p3 = f'{save_dir}/{stem}_masks.png'
    plt.savefig(p3, dpi=130, bbox_inches='tight'); plt.show()
    print(f'Saved: {p3}')

    # ── Metric table ──────────────────────────────────────────────────────
    print('\n' + '='*62)
    print(f'  METRICS — {image_name}')
    print('='*62)
    print(f'{"Metric":<22} {"Method A":>18} {"Method B":>18}  Winner')
    print('-'*62)
    for k in ['PSNR', 'SSIM', 'LPIPS', 'mask_coverage_%']:
        a, b = mA.get(k,'-'), mB.get(k,'-')
        if k == 'LPIPS':
            w = 'B' if b < a else ('A' if a < b else '=')
        elif k in ('PSNR','SSIM'):
            w = 'B' if b > a else ('A' if a > b else '=')
        else: w = ''
        print(f'{k:<22} {str(a):>18} {str(b):>18}  {w}')
    print(f'{"Time (s)":<22} {res_A["time_s"]:>18} {res_B["time_s"]:>18}')
    print('='*62)

## Cell 6 — Global Results Tracker

In [ ]:
all_results = []  # accumulates across all image uploads

def print_summary():
    if not all_results:
        print('No results yet.'); return
    print('\n' + '='*84)
    print('  AGGREGATE SUMMARY')
    print('='*84)
    print(f'{"Image":<25} {"N":>3} | {"A:PSNR":>8} {"B:PSNR":>8} | {"A:SSIM":>7} {"B:SSIM":>7} | {"A:LPIPS":>8} {"B:LPIPS":>8}')
    print('-'*84)
    tot = {'AP':0,'BP':0,'AS':0,'BS':0,'AL':0,'BL':0}
    for r in all_results:
        a,b = r['mA'], r['mB']
        print(f"{r['name'][:24]:<25} {r['n']:>3} | "
              f"{a['PSNR']:>8} {b['PSNR']:>8} | "
              f"{a['SSIM']:>7} {b['SSIM']:>7} | "
              f"{a['LPIPS']:>8} {b['LPIPS']:>8}")
        tot['AP']+=a['PSNR']; tot['BP']+=b['PSNR']
        tot['AS']+=a['SSIM']; tot['BS']+=b['SSIM']
        tot['AL']+=a['LPIPS']; tot['BL']+=b['LPIPS']
    n = len(all_results)
    print('-'*84)
    print(f'{"AVERAGE":<25} {"":>3} | '
          f'{tot["AP"]/n:>8.3f} {tot["BP"]/n:>8.3f} | '
          f'{tot["AS"]/n:>7.4f} {tot["BS"]/n:>7.4f} | '
          f'{tot["AL"]/n:>8.4f} {tot["BL"]/n:>8.4f}')
    print('='*84)
    print(f'  PSNR winner: {"B-Progressive" if tot["BP"]>tot["AP"] else "A-SingleShot"}')
    print(f'  SSIM winner: {"B-Progressive" if tot["BS"]>tot["AS"] else "A-SingleShot"}')
    print(f' LPIPS winner: {"B-Progressive" if tot["BL"]<tot["AL"] else "A-SingleShot"}')
    print('='*84)

## Cell 7 — Upload & Run (Re-run this cell for each new image)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
YOLO_CONF   = 0.4   # Lower if persons are missed
MAX_PERSONS = 8     # Cap to control runtime
SAVE_DIR    = 'results'

# ── Widgets ───────────────────────────────────────────────────────────────────
uploader = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp', multiple=False,
    description='Upload Image', layout=widgets.Layout(width='320px')
)
run_btn     = widgets.Button(description='Run Comparison',
                              button_style='success', layout=widgets.Layout(width='180px', height='38px'))
summary_btn = widgets.Button(description='Show Summary',
                              button_style='info',    layout=widgets.Layout(width='180px', height='38px'))
out = widgets.Output()

def on_run(b):
    with out:
        clear_output(wait=True)
        if not uploader.value:
            print('Please upload an image first!'); return

        uploaded   = list(uploader.value.values())[0]
        image_name = list(uploader.value.keys())[0]
        arr        = np.frombuffer(uploaded['content'], dtype=np.uint8)
        image_bgr  = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        image_rgb  = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        print(f'Image: {image_name} — {image_rgb.shape[1]}x{image_rgb.shape[0]}')

        print('\nDetecting persons...')
        persons = detect_persons(image_bgr, YOLO_CONF)[:MAX_PERSONS]
        print(f'Found {len(persons)} person(s)')
        if not persons:
            print('No persons detected. Try lowering YOLO_CONF.'); return

        print('\n--- METHOD A: SINGLE-SHOT ---')
        res_A = method_single_shot(image_rgb, persons)

        print('\n--- METHOD B: PROGRESSIVE ---')
        res_B = method_progressive(image_rgb, persons)

        print('\nComputing metrics...')
        mA = compute_metrics(image_rgb, res_A['result'], res_A['union_mask'])
        mB = compute_metrics(image_rgb, res_B['result'], res_B['union_mask'])

        show_results(image_name, image_rgb, persons, res_A, res_B, mA, mB, SAVE_DIR)

        all_results.append({'name': image_name, 'n': len(persons), 'mA': mA, 'mB': mB})
        print(f'\nTotal processed: {len(all_results)} image(s). Upload another or click Show Summary.')

def on_summary(b):
    with out:
        print_summary()

run_btn.on_click(on_run)
summary_btn.on_click(on_summary)

display(widgets.VBox([
    widgets.HTML('<b>Step 1: Upload an image</b>'),
    uploader,
    widgets.HTML('<b>Step 2: Choose action</b>'),
    widgets.HBox([run_btn, summary_btn]),
    out
]))

## Cell 8 — Final Aggregate Bar Chart (Run after all images)

In [ ]:
print_summary()

if len(all_results) >= 2:
    names   = [r['name'][:14] for r in all_results]
    psnr_A  = [r['mA']['PSNR']  for r in all_results]
    psnr_B  = [r['mB']['PSNR']  for r in all_results]
    ssim_A  = [r['mA']['SSIM']  for r in all_results]
    ssim_B  = [r['mB']['SSIM']  for r in all_results]
    lpips_A = [r['mA']['LPIPS'] for r in all_results]
    lpips_B = [r['mB']['LPIPS'] for r in all_results]

    x = np.arange(len(names)); w = 0.35
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Single-Shot vs Progressive — All Images', fontsize=13, fontweight='bold')

    for ax, vA, vB, title, ylab in [
        (axes[0], psnr_A,  psnr_B,  'PSNR (higher is better)', 'dB'),
        (axes[1], ssim_A,  ssim_B,  'SSIM (higher is better)', 'score'),
        (axes[2], lpips_A, lpips_B, 'LPIPS (lower is better)', 'score'),
    ]:
        ax.bar(x-w/2, vA, w, label='A: Single-Shot', color='#e07070', alpha=0.85)
        ax.bar(x+w/2, vB, w, label='B: Progressive', color='#5b9bd5', alpha=0.85)
        ax.set_title(title, fontweight='bold'); ax.set_ylabel(ylab)
        ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
        ax.legend(); ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/aggregate_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {SAVE_DIR}/aggregate_comparison.png')